# GTEx V11: transcriptomic transferability and molecular characteristics
**BMEN 600 | Public-reviewer Google Colab notebook**

**RQ:** To what extent can machine learning quantify transcriptomic transferability between human brain regions, and how are differences in transferability associated with molecular characteristics relevant to regional vulnerability in neurological disease?

This notebook analyzes GTEx V11 regional gene-read-count files to (1) evaluate out-of-sample cross-region predictions and (2) explore associations between gene-wise predictability and molecular characteristics. It does **not** train on Huntington's or Parkinson's patient samples.

**Reproduce the analysis:** Open this notebook in [Google Colab](https://colab.research.google.com/), set the **GitHub release owner** in Section 2 if it has not yet been configured by the project team, then choose **Runtime → Run all**. Section 2 downloads the public GitHub Release asset `gen_reads_GTEx.zip` automatically; there is no manual dataset upload.

The expected ZIP contains **13 `.gct.gz` regional gene-count files** directly or inside folders. The main analysis uses 12 brain regions; cervical spinal cord is excluded. Default mode runs six illustrative prediction directions; change `RUN_ALL_PAIRS=True` to evaluate every eligible direction (potentially much slower).

**Data/provenance:** The release must contain publicly redistributable GTEx data. Repository maintainers should record the original GTEx V11 download source, access date, release URL, and ideally a SHA-256 checksum. The code validates file counts and input formatting, but cross-study or clinical claims require additional work.


**Analysis scope:** The biological component tests associations between predictability and measured molecular features. Prespecified disease genes are included if expressed and variable in the training subset, with missing/excluded genes reported explicitly. It does not use disease labels or diagnose regional susceptibility. Pair comparisons are exploratory because cohorts and selected targets differ; repeated splits and common-cohort/common-gene analyses remain important validation steps.
**Verified revised run:** All six illustrative directions completed. Median gene-wise test R² ranged from 0.272 to 0.496 for approximately 1,000 training-selected targets plus eligible panel genes per pair. These values describe this split and target selection, not a definitive ranking. Seven graphs below were generated from these files.


## 1. Scientific rationale and analysis choices
Regional gene-expression patterns have been mapped in the Allen Human Brain Atlas [1], and reproducible transcriptional signatures have been related to anatomical and cellular organization [2]. GTEx provides a multi-tissue bulk RNA-seq resource [3]. These studies motivate the question; they do not establish that this notebook's prediction model will succeed.

Each independent case is a **donor**, not a gene or a tissue sample. Source and target regions must come from the same donor. A held-out donor's target measurements never enter fitting, feature selection, scaling, or tuning. Ridge regression provides regularization for correlated, high-dimensional predictors [5]; a separate inner validation split selects its penalty without using the test set [6].

We use log2(CPM + 1), calculated per sample from the complete count matrix. This is an exploratory normalization, **not TPM** and not TMM. Library-size scaling may retain compositional bias [4]; compare against TMM or another justified normalization before strong conclusions. These files alone lack age, sex, RIN, postmortem interval and batch metadata. Apparent transferability can reflect these factors or cell-type composition [3]. This analysis cannot establish neurodegenerative mechanisms, disease biomarkers or clinical utility.

BRONTE integrates GTEx with Allen Brain Atlas information to impute expression in additional tissues [7]. This notebook is a simpler pairwise ridge-regression analysis and does not reproduce BRONTE, use genotype data, or perform disease-risk inference. The cited BRONTE version is a preprint. Its imputation validation includes held-out measurements; avoid equating that design with this notebook's unseen-donor evaluation. Small prespecified disease panels [8]–[10] provide literature-grounded biological context, not comprehensive pathway or vulnerability signatures.

In [ ]:
from pathlib import Path
import gzip, json, hashlib, platform, itertools, shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from IPython.display import display

SEED = 42
RUN_ALL_PAIRS = False
INCLUDE_SPINAL_CORD = False
SOURCE = 'caudate_basal_ganglia'
TARGET = 'putamen_basal_ganglia'
MIN_PAIRED_DONORS = 60
N_SOURCE_GENES = 500
N_TARGET_GENES = 1000
ALPHAS = [1., 10., 100., 1000.]
N_BOOTSTRAPS = 1000
OUT = Path('/content/gtex_results')
OUT.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({'figure.dpi': 120, 'axes.spines.top': False, 'axes.spines.right': False})
# Prespecified illustrative panels, fixed before looking at test outcomes.
# PARK7 is also called DJ-1. These are disease-associated genes, not vulnerability labels.
DISEASE_PANELS = {
    'PD monogenic [8]': ['SNCA','LRRK2','VPS35','PRKN','PINK1','PARK7'],
    'AD familial [9]': ['APP','PSEN1','PSEN2'],
    'ALS common genes [10]': ['C9orf72','SOD1','TARDBP','FUS']}
PANEL_SYMBOLS = {g.upper() for genes in DISEASE_PANELS.values() for g in genes}
ILLUSTRATIVE_PAIRS = [
    ('caudate_basal_ganglia','putamen_basal_ganglia'),
    ('putamen_basal_ganglia','caudate_basal_ganglia'),
    ('cortex','frontal_cortex_ba9'), ('frontal_cortex_ba9','cortex'),
    ('cerebellum','cerebellar_hemisphere'), ('cerebellar_hemisphere','cerebellum')]

## 2. Automatically download the count files from GitHub Releases

The code below downloads the public Release asset **`gen_reads_GTEx.zip`** from GitHub, extracts its `.gct.gz` files to `/content/gtex_input`, and checks that **13 expected region files** were found. No reviewer uploads are required. **Repository maintainer:** set `GITHUB_OWNER` to your actual GitHub username or organization name before sharing the notebook; `BMEN600-Project` and release tag `v1.0-data` are the repository and tag names specified for this setup. If either differs, update them here. The ZIP must be attached under **Releases → v1.0-data → Assets**, not just committed in the repository.


In [ ]:
from pathlib import Path
import requests
import zipfile
import shutil

# REPOSITORY MAINTAINER: change GITHUB_OWNER to the actual public GitHub owner.
GITHUB_OWNER = "YOUR_USERNAME"
GITHUB_REPO = "BMEN600-Project"
RELEASE_TAG = "v1.0-data"
ZIP_NAME = "gen_reads_GTEx.zip"

if GITHUB_OWNER == "YOUR_USERNAME":
    raise ValueError(
        "Set GITHUB_OWNER to your actual GitHub username/organization, "
        "then commit the updated notebook so public reviewers can use Run all."
    )

DATA_URL = (
    f"https://github.com/{GITHUB_OWNER}/{GITHUB_REPO}/"
    f"releases/download/{RELEASE_TAG}/{ZIP_NAME}"
)
DATA_DIR = Path("/content/gtex_input")
DATA_DIR.mkdir(parents=True, exist_ok=True)
ZIP_PATH = Path("/content") / ZIP_NAME

print(f"Downloading GTEx V11 data from: {DATA_URL}")
try:
    with requests.get(DATA_URL, stream=True, timeout=(30, 180),
                      headers={"User-Agent": "GTEx-BMEN600-reproducibility-notebook"}) as response:
        response.raise_for_status()
        with ZIP_PATH.open("wb") as output:
            for chunk in response.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    output.write(chunk)
except requests.RequestException as exc:
    raise RuntimeError(
        "GitHub dataset download failed. Confirm that the repository and Release "
        "are public and that the Release contains the asset gen_reads_GTEx.zip. "
        f"Attempted URL: {DATA_URL}"
    ) from exc

# Extract expected data files only. Flatten optional directories inside ZIP.
# Do not extract arbitrary paths, and reject duplicate region filenames.
try:
    with zipfile.ZipFile(ZIP_PATH) as archive:
        members = [m for m in archive.infolist()
                   if not m.is_dir() and Path(m.filename).name.endswith(".gct.gz")]
        names = [Path(m.filename).name for m in members]
        if len(names) != len(set(names)):
            raise ValueError("ZIP contains duplicate .gct.gz filenames.")
        if len(names) != 13:
            raise ValueError(f"Expected 13 .gct.gz files in ZIP, found {len(names)}")
        unexpected = [name for name in names if not
                      (name.startswith("gene_reads_adult_gtex_v11_brain_") and
                       name.endswith(".gct.gz"))]
        if unexpected:
            raise ValueError(
                "Unexpected dataset filename(s): " + ", ".join(unexpected)
                + ". Ensure the files retain their original names (no '(1)' suffixes)."
            )
        for m in members:
            destination = DATA_DIR / Path(m.filename).name
            with archive.open(m) as source, destination.open("wb") as target:
                shutil.copyfileobj(source, target)
except (zipfile.BadZipFile, OSError, ValueError) as exc:
    raise RuntimeError("Unable to validate/extract GTEx release ZIP: " + str(exc)) from exc

paths = sorted(DATA_DIR.glob("gene_reads_adult_gtex_v11_brain_*.gct.gz"))
assert len(paths) == 13, f"Expected 13 region files after extraction, found {len(paths)}"
for path in paths:
    with path.open("rb") as f:
        if f.read(2) != b"\x1f\x8b":
            raise ValueError(f"Not a valid gzip file header: {path.name}")
print(f"Dataset ready: {len(paths)} region files downloaded and identified.")
print("Run Section 3 to load, validate, and normalize the count matrices.")


## 3. Load, validate and normalize
A GCT 1.2 file has a version line, a dimensions line, then the table. `Name` identifies genes and `Description` supplies symbols. GTEx donor IDs are the first two hyphen-separated fields of the sample ID. Multiple samples for one donor within a region are averaged **after sample-level normalization**; the summary records this choice. We retain full Ensembl IDs to avoid collisions.

In [ ]:
def load_region(path):
    with gzip.open(path, 'rt') as f:
        version = f.readline().strip()
        n_genes, n_samples = map(int, f.readline().split())
    if version != '#1.2':
        raise ValueError(f'Unsupported GCT version: {version}')
    table = pd.read_csv(path, sep='\t', skiprows=2)
    if table.shape != (n_genes, n_samples + 2):
        raise ValueError(f'Dimension mismatch: {path.name}')
    if table['Name'].duplicated().any():
        raise ValueError('Duplicate gene IDs')
    symbols = table.set_index('Name')['Description']
    counts = table.iloc[:, 2:].astype('float32')
    counts.index = table['Name']
    if not np.isfinite(counts.to_numpy()).all() or (counts.to_numpy() < 0).any():
        raise ValueError('Counts must be finite and nonnegative')
    libraries = counts.sum(axis=0)
    if (libraries <= 0).any():
        raise ValueError('Zero-count sample found')
    donors = []
    for sample in counts.columns:
        parts = sample.split('-')
        if len(parts) < 3 or parts[0] != 'GTEX':
            raise ValueError(f'Unexpected sample ID: {sample}')
        donors.append('-'.join(parts[:2]))
    logcpm = np.log2(counts.div(libraries, axis=1) * 1e6 + 1).T
    logcpm.index = donors
    matrix = logcpm.groupby(level=0, sort=True).mean().astype('float32')
    record = dict(region=path.name.removeprefix('gene_reads_adult_gtex_v11_brain_').removesuffix('.gct.gz'),
                  genes=n_genes, samples=n_samples, donors=len(matrix),
                  extra_samples=n_samples-len(matrix), median_library_reads=float(libraries.median()),
                  sha256=hashlib.sha256(path.read_bytes()).hexdigest())
    return matrix, symbols, record

regions, symbols, records = {}, {}, []
for path in paths:
    matrix, gene_symbols, record = load_region(path)
    regions[record['region']] = matrix
    symbols[record['region']] = gene_symbols
    records.append(record)
    print(record['region'], matrix.shape)
summary = pd.DataFrame(records)
summary.to_csv(OUT / 'table_1_dataset_summary.csv', index=False)
display(summary.drop(columns='sha256'))
active = sorted(r for r in regions if INCLUDE_SPINAL_CORD or not r.startswith('spinal_cord'))
overlap = pd.DataFrame([[len(regions[a].index.intersection(regions[b].index)) for b in active] for a in active], index=active, columns=active)
overlap.to_csv(OUT / 'donor_overlap.csv')

## 4. Dataset graphs
**Figure 1.** Unique donors per uploaded region after within-region aggregation. Counts describe available data, not the number used in every pair.

**Figure 2.** Donors with measurements in both regions. Each pair can have a different cohort. This limits direct comparison between pair scores.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
s = summary.sort_values('donors')
ax.barh(s.region.str.replace('_', ' '), s.donors, color='#357da8')
ax.set(xlabel='Unique donors', title='Figure 1. Available donors by region')
fig.tight_layout(); fig.savefig(OUT / 'figure_1_region_counts.png', dpi=200); plt.show()
fig, ax = plt.subplots(figsize=(11, 9))
im = ax.imshow(overlap, cmap='Blues')
labels = [r.replace('_', ' ') for r in active]
ax.set_xticks(range(len(active)), labels, rotation=90)
ax.set_yticks(range(len(active)), labels)
for i in range(len(active)):
    for j in range(len(active)):
        ax.text(j, i, str(overlap.iloc[i,j]), ha='center', va='center', fontsize=7)
ax.set_title('Figure 2. Paired donor availability')
fig.colorbar(im, ax=ax, label='Paired donors')
fig.tight_layout(); fig.savefig(OUT / 'figure_2_donor_overlap.png', dpi=200); plt.show()

## 5. Prediction and evaluation
For each direction, hold out 20% of paired donors. From the remaining donors, reserve an inner validation set to select alpha. Source and target genes are selected separately using only the inner training set: expression equivalent to CPM > 1 in at least 20% of donors, then highest variance. This fixed gene set is reused when fitting on all development donors. Scaling is fitted independently at each training stage.

The baseline predicts each target gene's development-set mean. The primary **prediction skill** is `1 − model MSE / baseline MSE`, where errors are averaged over donors and selected target genes. Zero matches the baseline, positive improves it, and negative is worse. This is baseline-relative skill, not conventional test R². Gene-wise test R² is exported separately. Bootstrap intervals resample held-out donors using fixed predictions; they quantify test-cohort uncertainty conditional on the fitted model, not training uncertainty. Do not treat genes as independent patients.

Different pairs use different selected targets and donors. The all-pair heatmap is exploratory; confirm rankings with repeated donor splits and common-target/common-cohort sensitivity analyses before interpreting pair differences biologically.

**R² matches the introduction:** For each selected target gene, `R² = 1 − Σ(y − ŷ)² / Σ(y − mean_test(y))²` over held-out donors. Report median gene-wise R² for each pair with a donor-bootstrap interval. Zero matches the gene's test-cohort mean, while negative values mean worse prediction; constant genes have undefined R². This differs from the development-mean baseline used for prediction skill. Neither score directly measures causal molecular sharing. Biological gene characteristics are calculated from development donors only, while predictability uses held-out donors. The small disease panels are included separately from variance-selected targets to prevent silently excluding every disease gene.

In [ ]:
def choose_genes(frame, n):
    expressed = (frame > 1).mean(axis=0) >= 0.2
    variance = frame.var(axis=0)
    ranked = variance[expressed & (variance > 1e-8)].sort_values(ascending=False)
    if len(ranked) < 10:
        raise ValueError('Too few expressed variable genes')
    return ranked.index[:n]

def analyze_pair(source, target):
    ids = regions[source].index.intersection(regions[target].index).sort_values()
    if len(ids) < MIN_PAIRED_DONORS:
        return None
    dev, test = train_test_split(np.arange(len(ids)), test_size=0.2, random_state=SEED)
    inner, val = train_test_split(dev, test_size=0.25, random_state=SEED)
    Xframe, Yframe = regions[source].loc[ids], regions[target].loc[ids]
    xgenes = choose_genes(Xframe.iloc[inner], N_SOURCE_GENES)
    ygenes = choose_genes(Yframe.iloc[inner], N_TARGET_GENES)
    # Add prespecified disease genes only when eligible using training data.
    eligible = ((Yframe.iloc[inner] > 1).mean(axis=0) >= 0.2) & (Yframe.iloc[inner].var(axis=0) > 1e-8)
    panel_ids = symbols[target].index[symbols[target].astype(str).str.upper().isin(PANEL_SYMBOLS)]
    panel_ids = [g for g in panel_ids if g in eligible.index and eligible.loc[g]]
    ygenes = ygenes.union(pd.Index(panel_ids), sort=False)
    X = Xframe[xgenes].to_numpy(dtype='float64')
    Y = Yframe[ygenes].to_numpy(dtype='float64')
    scaler = StandardScaler().fit(X[inner])
    Xi, Xv = scaler.transform(X[inner]), scaler.transform(X[val])
    validation = []
    for alpha in ALPHAS:
        fit = Ridge(alpha=alpha, solver='cholesky').fit(Xi, Y[inner])
        validation.append(np.mean((Y[val] - fit.predict(Xv))**2))
    best = ALPHAS[int(np.argmin(validation))]
    scaler = StandardScaler().fit(X[dev])
    model = Ridge(alpha=best, solver='cholesky').fit(scaler.transform(X[dev]), Y[dev])
    truth = Y[test]
    pred = model.predict(scaler.transform(X[test]))
    baseline = np.broadcast_to(Y[dev].mean(axis=0), truth.shape)
    model_loss = ((truth-pred)**2).mean(axis=1)
    base_loss = ((truth-baseline)**2).mean(axis=1)
    skill = 1-model_loss.mean()/base_loss.mean()
    rng = np.random.default_rng(SEED)
    sampled = rng.integers(0, len(test), size=(N_BOOTSTRAPS, len(test)))
    bs = 1-model_loss[sampled].mean(axis=1)/base_loss[sampled].mean(axis=1)
    lo, hi = np.quantile(bs, [0.025, 0.975])
    sse = ((truth-pred)**2).sum(axis=0)
    sst = ((truth-truth.mean(axis=0))**2).sum(axis=0)
    r2 = np.divide(sse, sst, out=np.full_like(sse, np.nan), where=sst>1e-12)
    gene_table = pd.DataFrame({'gene_id':ygenes, 'symbol':symbols[target].reindex(ygenes).values,
                               'test_r2':1-r2, 'model_mse':((truth-pred)**2).mean(axis=0),
                               'baseline_mse':((truth-baseline)**2).mean(axis=0),
                               'target_dev_mean':Yframe.iloc[dev][ygenes].mean().values,
                               'target_dev_variance':Yframe.iloc[dev][ygenes].var().values,
                               'source_dev_mean':Xframe.iloc[dev].reindex(columns=ygenes).mean().values})
    gene_table['abs_regional_mean_difference'] = abs(gene_table.target_dev_mean-gene_table.source_dev_mean)
    gene_table['gene_skill'] = 1-gene_table.model_mse/gene_table.baseline_mse
    gene_table['source'], gene_table['target'] = source, target
    # Conditional bootstrap of median gene-wise R², resampling donors, not genes.
    r2_boot = []
    for sample in sampled:
        t, pr = truth[sample], pred[sample]
        den = ((t-t.mean(axis=0))**2).sum(axis=0)
        rr = 1-np.divide(((t-pr)**2).sum(axis=0),den,out=np.full_like(den,np.nan),where=den>1e-12)
        r2_boot.append(np.nanmedian(rr))
    r2lo,r2hi = np.nanquantile(r2_boot,[0.025,0.975])
    prefix = source+'__to__'+target
    gene_table.to_csv(OUT / (prefix+'_genes.csv'), index=False)
    # Keep donor-level files local; avoid sharing IDs in public repositories.
    pd.DataFrame({'donor_id':ids[test], 'model_mse':model_loss, 'baseline_mse':base_loss}).to_csv(OUT / (prefix+'_test_errors_PRIVATE.csv'), index=False)
    pd.DataFrame({'donor_id':ids, 'split':['test' if i in test else 'development' for i in range(len(ids))]}).to_csv(OUT / (prefix+'_splits_PRIVATE.csv'), index=False)
    row = dict(source=source,target=target,n_paired=len(ids),n_development=len(dev),n_test=len(test),
               n_source_genes=len(xgenes),n_target_genes=len(ygenes),alpha=best,
               model_mse=model_loss.mean(),baseline_mse=base_loss.mean(),
               skill=skill,ci_low=lo,ci_high=hi,median_gene_r2=float(np.nanmedian(1-r2)),
               r2_ci_low=float(r2lo),r2_ci_high=float(r2hi))
    return row, dict(truth=truth,pred=pred,baseline=baseline,genes=gene_table)

pairs = list(itertools.permutations(active, 2)) if RUN_ALL_PAIRS else [(a,b) for a,b in ILLUSTRATIVE_PAIRS if a in active and b in active]
if not pairs and SOURCE in active and TARGET in active: pairs = [(SOURCE,TARGET)]
results, skipped, example, gene_results = [], [], None, []
for source,target in pairs:
    if source not in active or target not in active:
        raise ValueError('Selected region unavailable or excluded')
    output = analyze_pair(source,target)
    if output is None:
        skipped.append({'source':source,'target':target,'reason':'too few paired donors'})
        continue
    row, detail = output
    results.append(row)
    gene_results.append(detail['genes'])
    if example is None:
        example = (row,detail)
    print(source,'→',target, 'skill:',round(row['skill'],3))
if not results:
    raise ValueError('No eligible pairs. Inspect overlap and minimum donor threshold.')
results = pd.DataFrame(results)
results.to_csv(OUT / 'table_2_prediction_results.csv',index=False)
pd.DataFrame(skipped).to_csv(OUT / 'skipped_pairs.csv', index=False)
display(results)
all_genes = pd.concat(gene_results,ignore_index=True)
all_genes.to_csv(OUT / 'table_3_gene_transferability_features.csv',index=False)

## 6. Result graphs
**Figure 3.** Prediction skill with conditional 95% donor-bootstrap intervals. Zero indicates the mean-expression baseline. An interval crossing zero does not show clear improvement for this held-out cohort.

**Figure 4.** Predicted versus observed expression for the example target gene with greatest variance in the development donors (not selected for best test performance). Each point is one test donor. The dashed line is perfect prediction.

**Figure 5 (all-pair mode).** Directional prediction skill. Rows are sources, columns are targets; missing pairs remain blank. Pairwise rankings require the sensitivity analyses described above.

In [ ]:
r = results.sort_values('skill')
fig, ax = plt.subplots(figsize=(10,max(3,0.3*len(r))))
y = np.arange(len(r))
ax.hlines(y,r.ci_low,r.ci_high,color='#357da8')
ax.scatter(r.skill,y,color='#163b54')
ax.set_yticks(y, [a.replace('_',' ')+' → '+b.replace('_',' ') for a,b in zip(r.source,r.target)])
ax.axvline(0,color='gray',linestyle='--')
ax.set(xlabel='Skill relative to target-mean baseline',title='Figure 3. Cross-region prediction skill (95% intervals)')
fig.tight_layout(); fig.savefig(OUT / 'figure_3_prediction_skill.png',dpi=200); plt.show()
row, detail = example
# Target genes are ordered by development-inner-training variance.
j = 0
truth, pred = detail['truth'][:,j], detail['pred'][:,j]
fig, ax = plt.subplots(figsize=(5,5))
ax.scatter(truth,pred,alpha=0.8,color='#357da8')
limits = [min(truth.min(),pred.min()),max(truth.max(),pred.max())]
ax.plot(limits,limits,'--',color='gray')
name = detail['genes'].iloc[j]['symbol']
ax.set(xlabel='Observed log2(CPM + 1)',ylabel='Predicted log2(CPM + 1)',title=f'Figure 4. Held-out donors: {name}')
fig.tight_layout(); fig.savefig(OUT / 'figure_4_example_gene.png',dpi=200); plt.show()
if RUN_ALL_PAIRS:
    heat = results.pivot(index='source',columns='target',values='skill').reindex(index=active,columns=active)
    fig, ax = plt.subplots(figsize=(11,9))
    bound = max(1,float(np.nanmax(np.abs(heat.to_numpy()))))
    im = ax.imshow(np.ma.masked_invalid(heat.to_numpy()),cmap='RdBu',vmin=-bound,vmax=bound)
    ax.set_xticks(range(len(active)),labels,rotation=90)
    ax.set_yticks(range(len(active)),labels)
    ax.set(xlabel='Target',ylabel='Source',title='Figure 5. Directional prediction skill')
    fig.colorbar(im,ax=ax,label='Prediction skill')
    fig.tight_layout(); fig.savefig(OUT / 'figure_5_transferability_heatmap.png',dpi=200); plt.show()

## 7. Molecular characteristics associated with transferability
**Figure 6:** Median gene-wise test R² and conditional 95% donor-bootstrap intervals by direction.

**Figure 7:** For the first evaluated pair, gene-wise test R² versus development-set regional mean-expression difference, coloured by target expression level. The CSV also reports within-pair Spearman associations for expression level, variance and regional difference. These are descriptive associations among correlated genes, with no independent-gene significance claim; expression variance and R² are mathematically related and may reflect measurement reliability.

**Figure 8:** Gene-wise test R² for prespecified disease-associated genes across evaluated pairs [8]–[10]. Blank entries mean not evaluated or undefined, not low transferability. A coverage table identifies genes absent from the files or excluded by the training expression/variance filter. The panels are small and incomplete; do not call them pathway enrichment or disease-validation results.

In [ ]:
# Pair-level R² overview
r = results.sort_values('median_gene_r2')
fig,ax = plt.subplots(figsize=(10,max(3,0.35*len(r))))
y = np.arange(len(r))
ax.hlines(y,r.r2_ci_low,r.r2_ci_high,color='#357da8')
ax.scatter(r.median_gene_r2,y,color='#163b54')
ax.set_yticks(y,[a.replace('_',' ')+' → '+b.replace('_',' ') for a,b in zip(r.source,r.target)])
ax.axvline(0,color='gray',linestyle='--')
ax.set(xlabel='Median gene-wise test R²',title='Figure 6. Transferability across region pairs')
fig.tight_layout();fig.savefig(OUT/'figure_6_gene_r2.png',dpi=200);plt.show()

features = ['target_dev_mean','target_dev_variance','abs_regional_mean_difference']
associations=[]
for (source,target), group in all_genes.groupby(['source','target']):
    for feature in features:
        valid=group[[feature,'test_r2']].replace([np.inf,-np.inf],np.nan).dropna()
        rho=valid[feature].rank().corr(valid.test_r2.rank()) if len(valid)>2 else np.nan
        associations.append(dict(source=source,target=target,feature=feature,n_genes=len(valid),spearman_rho=rho))
associations=pd.DataFrame(associations)
associations.to_csv(OUT/'table_4_molecular_associations.csv',index=False)
display(associations)
first=all_genes[(all_genes.source==results.iloc[0].source)&(all_genes.target==results.iloc[0].target)]
fig,ax=plt.subplots(figsize=(7,5))
im=ax.scatter(first.abs_regional_mean_difference,first.test_r2,c=first.target_dev_mean,cmap='viridis',s=15,alpha=0.7)
ax.axhline(0,color='gray',linestyle='--')
ax.set(xlabel='Absolute regional difference in mean log2(CPM + 1)',ylabel='Gene-wise test R²',title='Figure 7. Molecular characteristics and predictability')
fig.colorbar(im,ax=ax,label='Target development mean log2(CPM + 1)')
fig.tight_layout();fig.savefig(OUT/'figure_7_molecular_features.png',dpi=200);plt.show()

# Coverage remains explicit even if a disease gene is not evaluated.
coverage=[]
for _, pair in results.iterrows():
    target=pair.target
    evaluated=all_genes[(all_genes.source==pair.source)&(all_genes.target==target)]
    present=set(symbols[target].astype(str).str.upper())
    evaluated_symbols=set(evaluated.symbol.astype(str).str.upper())
    for panel, genes in DISEASE_PANELS.items():
        for gene in genes:
            status=('evaluated' if gene.upper() in evaluated_symbols else
                    'excluded by training filter' if gene.upper() in present else 'absent from input annotation')
            coverage.append(dict(source=pair.source,target=target,panel=panel,gene=gene,status=status))
coverage=pd.DataFrame(coverage)
coverage.to_csv(OUT/'table_5_disease_gene_coverage.csv',index=False)
disease=all_genes[all_genes.symbol.astype(str).str.upper().isin(PANEL_SYMBOLS)].copy()
disease['symbol_upper']=disease.symbol.astype(str).str.upper()
disease['pair']=disease.source.str.replace('_',' ')+' → '+disease.target.str.replace('_',' ')
disease.to_csv(OUT/'table_6_disease_gene_transferability.csv',index=False)
display(coverage.groupby(['panel','status']).size().rename('gene-pair entries').reset_index())
if not disease.empty:
    heat=disease.pivot_table(index='symbol_upper',columns='pair',values='test_r2',aggfunc='median')
    order=[g.upper() for genes in DISEASE_PANELS.values() for g in genes]
    heat=heat.reindex(index=order)
    fig,ax=plt.subplots(figsize=(max(8,0.4*len(heat.columns)),7))
    im=ax.imshow(np.ma.masked_invalid(heat.to_numpy()),cmap='RdBu',vmin=-1,vmax=1,aspect='auto')
    ax.set_xticks(range(len(heat.columns)),heat.columns,rotation=90,fontsize=7)
    ax.set_yticks(range(len(heat)),heat.index)
    ax.set_title('Figure 8. Disease-associated gene predictability (exploratory)')
    fig.colorbar(im,ax=ax,label='Test R²; colours clipped at −1 and 1')
    fig.tight_layout();fig.savefig(OUT/'figure_8_disease_gene_r2.png',dpi=200);plt.show()
else:
    print('No panel genes evaluated; inspect coverage. No biological conclusion is supported.')

## 8. Interpretation and project reporting
Use `table_1_dataset_summary.csv` and Figures 1–2 for the dataset description. Use `table_2_prediction_results.csv` and Figure 3 for the primary result and baseline comparison. Gene-level CSVs and Figures 6–8 support the molecular-characteristics component of the RQ. Report both median gene-wise R² and baseline-relative skill. Molecular-feature associations and disease-panel patterns are hypothesis-generating; overlapping donors across pairs prevent treating pair scores as independent cases.

Report actual results, including negative skill. High performance supports predictable cross-region variation in this design; it does not establish a causal molecular connection. Age, sex, tissue quality, batch and cell mixture may explain part of the prediction. GTEx tissue donors are not an HD/PD case-control cohort. No disease data were attached, so disease validation is not included.

Before the final paper: add verified GTEx V11 provenance/access date and data-use terms; obtain permitted sample/donor covariates; examine normalization sensitivity; repeat donor splits; compare pairs on common genes and, where feasible, the same donors; distinguish exploratory gene rankings from confirmed biological pathways. Follow GTEx privacy and sharing rules. Do not publish donor-level outputs or original expression matrices without verifying terms.

For BMEN 600: number and caption figures, explain the baseline and uncertainty, acknowledge model and cohort limitations, include this notebook in your GitHub repository, record the submitted commit, and add author contributions and an AI-use statement. Suggested AI statement to adapt truthfully: “ChatGPT assisted with notebook construction and method explanations. Team members verified references, reviewed donor splitting and preprocessing, ran the analysis, and checked outputs.”

In [ ]:
metadata = {'question':'ML transcriptomic transferability and disease-relevant molecular characteristics', 'seed':SEED,
            'run_all_pairs':RUN_ALL_PAIRS,'include_spinal_cord':INCLUDE_SPINAL_CORD,
            'normalization':'log2(CPM+1); total gene counts per sample',
            'source_genes':N_SOURCE_GENES,'target_genes':N_TARGET_GENES,
            'alphas':ALPHAS,'bootstraps':N_BOOTSTRAPS,
            'python':platform.python_version(),'numpy':np.__version__,
            'pandas':pd.__version__,'sklearn':sklearn.__version__,
            'input_files':records,'disease_panels':DISEASE_PANELS,'illustrative_pairs':ILLUSTRATIVE_PAIRS}
(OUT/'analysis_metadata.json').write_text(json.dumps(metadata,indent=2))
# Public export omits files labelled PRIVATE. Keep full local results for verification.
PUBLIC = Path('/content/gtex_public_results')
PUBLIC.mkdir(exist_ok=True)
for old in PUBLIC.iterdir():
    if old.is_file(): old.unlink()
for p in OUT.iterdir():
    if p.is_file() and 'PRIVATE' not in p.name:
        shutil.copy2(p,PUBLIC/p.name)
archive = shutil.make_archive('/content/gtex_results_public','zip',PUBLIC)
from google.colab import files
files.download(archive)

## 9. References (IEEE-style)
[1] M. J. Hawrylycz et al., “An anatomically comprehensive atlas of the adult human brain transcriptome,” *Nature*, vol. 489, pp. 391–399, 2012. doi: [10.1038/nature11405](https://doi.org/10.1038/nature11405).

[2] M. Hawrylycz et al., “Canonical genetic signatures of the adult human brain,” *Nature Neuroscience*, vol. 18, pp. 1832–1844, 2015. doi: [10.1038/nn.4171](https://doi.org/10.1038/nn.4171).

[3] GTEx Consortium, “The GTEx Consortium atlas of genetic regulatory effects across human tissues,” *Science*, vol. 369, pp. 1318–1330, 2020. doi: [10.1126/science.aaz1776](https://doi.org/10.1126/science.aaz1776). Background publication for GTEx; it describes V8, **not** the supplied V11 release. Add the verified V11 release citation/provenance separately.

[4] M. D. Robinson and A. Oshlack, “A scaling normalization method for differential expression analysis of RNA-seq data,” *Genome Biology*, vol. 11, R25, 2010. doi: [10.1186/gb-2010-11-3-r25](https://doi.org/10.1186/gb-2010-11-3-r25). Supports discussion of compositional bias; TMM is not implemented here.

[5] A. E. Hoerl and R. W. Kennard, “Ridge regression: Biased estimation for nonorthogonal problems,” *Technometrics*, vol. 12, no. 1, pp. 55–67, 1970. doi: [10.1080/00401706.1970.10488634](https://doi.org/10.1080/00401706.1970.10488634).

[6] S. Varma and R. Simon, “Bias in error estimation when using cross-validation for model selection,” *BMC Bioinformatics*, vol. 7, art. 91, 2006. doi: [10.1186/1471-2105-7-91](https://doi.org/10.1186/1471-2105-7-91). Supports separating tuning from evaluation; this notebook uses separate validation/test splits rather than nested cross-validation.

Software: [scikit-learn Ridge](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Ridge.html), [StandardScaler](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html).

[7] J. Ke, T. Ge, and R. D. Melamed, “Discovering disease genetic variation impacting gene expression in 103 brain tissues with the Brain Ontology Expression (BRONTE) graph neural network model,” *bioRxiv*, preprint, May 2025. doi: [10.1101/2025.05.08.652857](https://doi.org/10.1101/2025.05.08.652857). The cited version is not peer reviewed.

[8] F. Jia, A. Fellner, and K. R. Kumar, “Monogenic Parkinson's disease: Genotype, phenotype, pathophysiology, and genetic testing,” *Genes*, vol. 13, no. 3, art. 471, 2022. doi: [10.3390/genes13030471](https://doi.org/10.3390/genes13030471). Supports the six-gene PD panel; gene association does not establish regional vulnerability.

[9] “Alzheimer Disease Overview,” *GeneReviews*, University of Washington, Seattle. [NCBI Bookshelf](https://www.ncbi.nlm.nih.gov/books/NBK1161/), accessed Oct. 9, 2026. Supports APP/PSEN1/PSEN2 familial AD context; this is a clinical reference, not an original cohort study.

[10] Z.-Y. Zou et al., “Genetic epidemiology of amyotrophic lateral sclerosis: A systematic review and meta-analysis,” *Journal of Neurology, Neurosurgery & Psychiatry*, vol. 88, no. 7, pp. 540–549, 2017. doi: [10.1136/jnnp-2016-315018](https://doi.org/10.1136/jnnp-2016-315018). Supports the four-gene ALS panel; repeat expansions or mutations are not measured by this expression analysis.